In [ ]:
import yaml
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

with open(os.path.join(PROJECT_ROOT, "config", "base2.yaml"), "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

emb_name = cfg["embedding"]["use"] 
emb_conf = cfg["embedding"]["siliconflow"]
print("Using embedding:", emb_name)
print("Embedding config:", emb_conf)


In [ ]:
import requests

def embed_texts_siliconflow(texts, emb_conf):
    url = emb_conf["api_base"] + "/embeddings" 
    api_key = emb_conf["api_key"]
    model = emb_conf["model"]

    payload = {
        "model": model,
        "input": texts
    }
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    r = requests.post(url, json=payload, headers=headers)
    r.raise_for_status()
    data = r.json()

    # 返回一批 embedding
    return [item["embedding"] for item in data["data"]]

import yaml
from camel.models import OpenAIModel

def load_llm_client(config_path):
    """
    根据你的 YAML 加载对应模型的 (api_key, api_base, model)
    并构造 openai SDK 的客户端。
    """
    with open(config_path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    llm_use = cfg["llm"]["use"]
    provider_cfg = cfg["llm"][llm_use]

    # 支持 key 池：如果是 list，则取第一个 key（可改成轮询）
    if isinstance(provider_cfg, list):
        provider_cfg = provider_cfg[0]

    model_name = provider_cfg["model"]
    api_key = provider_cfg["api_key"]
    api_base = provider_cfg["api_base"]  # 你 YAML 的字段名

    # Camel-AI 要求 url=xxx
    url = api_base.rstrip("/")  # 去掉结尾斜杠避免重复

    # 构建 model 配置字典（过滤 None）
    model_config = {
        "temperature": provider_cfg.get("temperature"),
        "top_p": provider_cfg.get("top_p"),
        "max_tokens": provider_cfg.get("max_tokens")
    }
    model_config = {k: v for k, v in model_config.items() if v is not None}

    model = OpenAIModel(
        model_type=model_name,       
        model_config_dict=model_config,
        api_key=api_key,
        url=url                     
    )

    return model


In [ ]:
import faiss
import json

# 加载 index
index = faiss.read_index(os.path.join(PROJECT_ROOT, "data", "faiss", "faiss2.index"))

# 加载 docstore（你的 refined_document_chunks.json）
with open(os.path.join(PROJECT_ROOT, "data", "faiss", "refined_document_chunks.json"), "r", encoding="utf-8") as f:
    docstore = json.load(f)

print("Index loaded:", index.ntotal)
print("Docstore loaded:", len(docstore))


In [ ]:
import numpy as np

def embed_query(text, emb_conf):
    vec = embed_texts_siliconflow([text], emb_conf)[0]
    return np.array(vec, dtype="float32").reshape(1, -1)

def get_doc_by_id(chunk_id, docstore):
    for item in docstore:
        if item["chunk_id"] == int(chunk_id):
            return item
    return None

In [ ]:
query = "给我一些dfs的例题？"  

query_vec = embed_query(query, emb_conf)

# 检索 top 10
D, I = index.search(query_vec, k=10)

print("Search result IDs:", I[0])

In [ ]:
results = []

for cid in I[0]:
    if cid == -1:
        continue

    doc = get_doc_by_id(cid, docstore)
    if doc:
        results.append(doc)

results

In [ ]:
from camel.agents import ChatAgent

context = "\n\n".join(
    f"[{r['title']}] {r['content']}"
    for r in results
)

model = load_llm_client(config_path=os.path.join(PROJECT_ROOT, "", "config", "base2.yaml"))
system_prompt = "你是一名专业教材问答助手，请根据下列上下文回答问题："

user_prompt = f"""
问题：{query}

参考内容：
{context}

请给出清晰、准确、简明的回答：
"""

agent = ChatAgent(system_prompt, model=model)

response = agent.step(user_prompt).msg.content.strip()
response

In [ ]:
print(user_prompt)

In [ ]:
import jieba
from rank_bm25 import BM25Okapi

# 1. 构建 corpus：用 content（也可以拼上 title）
corpus_texts = []
chunk_ids = []

for doc in docstore:
    text = doc["content"]
    # 你也可以：text = doc["title"] + " " + doc["content"]
    corpus_texts.append(text)
    chunk_ids.append(doc["chunk_id"])

# 2. 做中文分词
tokenized_corpus = [list(jieba.cut(text)) for text in corpus_texts]

# 3. 构建 BM25 模型
bm25 = BM25Okapi(tokenized_corpus)

print("BM25 corpus size:", len(tokenized_corpus))


In [ ]:
def bm25_search(query, top_k=20):
    """
    使用 BM25 进行关键词检索，返回 [(chunk_id, score), ...]
    """
    query_tokens = list(jieba.cut(query))
    scores = bm25.get_scores(query_tokens)
    
    # 取 top_k 的索引
    import numpy as np
    top_k = min(top_k, len(scores))
    top_indices = np.argsort(scores)[::-1][:top_k]  # 按得分从高到低
    
    results = []
    for idx in top_indices:
        cid = chunk_ids[idx]
        results.append((cid, float(scores[idx])))
    return results


In [ ]:
bm25_results = bm25_search("算法复杂性", top_k=5)
for cid, score in bm25_results:
    doc = get_doc_by_id(cid, docstore)
    print("Chunk ID:", cid)
    print("Score:", score)
    print("Title:", doc["title"])
    print("Path:", " > ".join(doc["path_titles"]))
    print("Content snippet:", doc["content"][:120], "...")
    
    print("="*60)

In [ ]:
import json
from collections import defaultdict

KG_JSON_PATH = os.path.join(PROJECT_ROOT, "data", "knowledge_graph", "test_new.json")
ID2CHUNK_PATH = os.path.join(PROJECT_ROOT, "data", "knowledge_graph", "id2chunk.json")

with open(KG_JSON_PATH, "r", encoding="utf-8") as f:
    kg_edges = json.load(f)

with open(ID2CHUNK_PATH, "r", encoding="utf-8") as f:
    id2chunk = json.load(f)

print("✅ KG 三元组:", len(kg_edges))
print("✅ id2chunk 数量:", len(id2chunk))

In [ ]:
from collections import defaultdict

entity2chunk = {}
term2chunks = defaultdict(set)

for edge in kg_edges:
    s = edge["start_node"]
    t = edge["end_node"]

    s_name = s["properties"].get("name")
    s_cid  = s["properties"].get("chunk id")

    t_label = t["label"]
    t_name  = t["properties"].get("name")
    t_cid   = t["properties"].get("chunk id")

    # ==========实体 → chunk ==========
    if isinstance(s_name, str) and s_cid:
        entity2chunk[s_name] = s_cid
        term2chunks[s_name].add(s_cid)

    # ==========attribute / keyword → chunk ==========
    if t_name and t_cid:

        # 正常字符串
        if isinstance(t_name, str):
            term2chunks[t_name].add(t_cid)

        # list
        elif isinstance(t_name, list):
            for name_item in t_name:
                if isinstance(name_item, str):
                    term2chunks[name_item].add(t_cid)

print("✅ 实体索引数:", len(entity2chunk))
print("✅ 词项索引数(term2chunks):", len(term2chunks))


In [ ]:
def kg_simple_retrieve(query, top_k=10):
    hit_chunks = defaultdict(int)

    for term, cids in term2chunks.items():
        if term in query:
            for cid in cids:
                hit_chunks[cid] += 1  # 命中次数作为简单打分

    # 按命中次数排序
    sorted_hits = sorted(hit_chunks.items(), key=lambda x: x[1], reverse=True)
    sorted_hits = sorted_hits[:top_k]

    chunk_ids = [cid for cid, _ in sorted_hits]
    return chunk_ids

In [ ]:
def fetch_chunks_by_ids(chunk_ids, max_len=1000):
    chunks = []
    for cid in chunk_ids:
        if cid in id2chunk:
            text = id2chunk[cid]
            chunks.append(text[:max_len])  # 防止太长
    return chunks

In [ ]:
def kg_rag_simple(query, top_k=10):
    chunk_ids = kg_simple_retrieve(query, top_k=top_k)
    chunks = fetch_chunks_by_ids(chunk_ids)

    return {
        "query": query,
        "chunk_ids": chunk_ids,
        "chunks": chunks
    }

In [ ]:
query = "动态规划是什么"
res = kg_rag_simple(query, top_k=5)

print("✅ 命中的 chunk_id:")
print(res["chunk_ids"])

print("\n✅ 命中的原文证据:")
for i, c in enumerate(res["chunks"]):
    print(f"\n【证据 {i+1}】")
    print(c[:500])